# 断点标签质量审查

## tl;dr

252 条单字 + 200 条普通样本完成 AI 初审。单字组 137 条可接受、4 条明确问题、111 条不确定。原标点标签有噪声，全部删单字会同时删掉有效监督。**不是人工 gold；事后预测诊断单独统计。**

## Context & Methods

种子 2026100417；普通样本从固定100k验证其余99748条中无放回抽200。先隐藏预测作AI初审，冻结后关联4组已存预测。29条原标签可接受但新增失配的样本另作可见预测的诊断。

### Key Assumptions

A不等于唯一答案；M替代位置不穷尽；U不算错。没有恢复原文。前文已有部分例子暴露，非严格独立盲审。不同采样比例不可混合外推。模型是单种子继续训练，并非从零未见单字。

In [1]:
from pathlib import Path
import sys, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'training/audit_boundary_labels.py').exists())
sys.path.insert(0, str(root / 'training'))
from audit_boundary_labels import analyze, OUT, REVIEW


## Data

使用冻结的 sample.jsonl、annotations.jsonl、freeze.json 和两份包含四组预测的 NPZ。重跑依赖原有 ignored smoke artifacts。脚本验证源文件 SHA、标注覆盖与预测 targets 顺序。安装依赖可在独立环境使用 `pip install nbformat nbclient ipykernel numpy`。

In [2]:
result = analyze()
assert all(result['checks'].values())
print(json.dumps(result['checks'], indent=2, ensure_ascii=False))

{
  "validation_sha256_matches": true,
  "sample_selection_reproduced": true,
  "annotation_ids_and_cuts_valid": true,
  "annotation_freeze_matches": true,
  "four_prediction_arrays_match_validation_order": true,
  "prior_252_counts_reconciled": true,
  "posthoc_coverage_and_prediction_cuts_match": true
}


## Results

### 1. 冻结标签分布

In [3]:
for group in result['review_counts']:
    assert sum(group[k] for k in ['A','M','E','U']) == group['n']
    print(group)

{'cohort': 'single', 'n': 252, 'documents': 200, 'A': 133, 'M': 4, 'E': 4, 'U': 111}
{'cohort': 'ordinary', 'n': 200, 'documents': 198, 'A': 133, 'M': 50, 'E': 0, 'U': 17}


### 2. 单字组精确标签匹配变化

每行分母按初审类别定义；精确失配不等于语义错误。

In [4]:
for row in result['exact_gold_scores']:
    assert row['after'] - row['before'] == row['newly_right'] - row['newly_wrong']
    if row['cohort'] == 'single':
        print({k:v for k,v in row.items() if k not in ['lost_ids','gained_ids']})

{'cohort': 'single', 'mode': 'inverse-cell', 'verdict': 'all', 'n': 252, 'before': 203, 'after': 164, 'newly_wrong': 39, 'newly_right': 0}
{'cohort': 'single', 'mode': 'inverse-cell', 'verdict': 'A', 'n': 133, 'before': 117, 'after': 96, 'newly_wrong': 21, 'newly_right': 0}
{'cohort': 'single', 'mode': 'inverse-cell', 'verdict': 'M', 'n': 4, 'before': 4, 'after': 3, 'newly_wrong': 1, 'newly_right': 0}
{'cohort': 'single', 'mode': 'inverse-cell', 'verdict': 'E', 'n': 4, 'before': 1, 'after': 0, 'newly_wrong': 1, 'newly_right': 0}
{'cohort': 'single', 'mode': 'inverse-cell', 'verdict': 'U', 'n': 111, 'before': 81, 'after': 65, 'newly_wrong': 16, 'newly_right': 0}
{'cohort': 'single', 'mode': 'none', 'verdict': 'all', 'n': 252, 'before': 179, 'after': 153, 'newly_wrong': 26, 'newly_right': 0}
{'cohort': 'single', 'mode': 'none', 'verdict': 'A', 'n': 133, 'before': 105, 'after': 92, 'newly_wrong': 13, 'newly_right': 0}
{'cohort': 'single', 'mode': 'none', 'verdict': 'M', 'n': 4, 'before': 

### 3. 新增失配中的预测质量

只包含原A/M、且从匹配原标签变成失配的29条去重样本。下列是事后AI诊断，不能当独立盲测。

In [5]:
for row in result['unblinded_followup']['by_mode']:
    print(row)
for row in result['multiple_boundary_scores']:
    assert row['exact'] <= row['listed_acceptable'] <= row['n']
    if row['alternative_hit_ids']:
        print(row)

{'mode': 'inverse-cell', 'E': 7, 'U': 7, 'A': 8}
{'mode': 'none', 'E': 5, 'A': 6, 'U': 3}
{'cohort': 'single', 'model': 'none_before', 'n': 4, 'exact': 3, 'listed_acceptable': 4, 'alternative_hit_ids': ['S174']}
{'cohort': 'single', 'model': 'inverse-cell_after', 'n': 4, 'exact': 3, 'listed_acceptable': 4, 'alternative_hit_ids': ['S174']}
{'cohort': 'single', 'model': 'none_after', 'n': 4, 'exact': 2, 'listed_acceptable': 4, 'alternative_hit_ids': ['S012', 'S174']}
{'cohort': 'ordinary', 'model': 'inverse-cell_before', 'n': 50, 'exact': 44, 'listed_acceptable': 46, 'alternative_hit_ids': ['N076', 'N194']}
{'cohort': 'ordinary', 'model': 'none_before', 'n': 50, 'exact': 43, 'listed_acceptable': 45, 'alternative_hit_ids': ['N076', 'N194']}
{'cohort': 'ordinary', 'model': 'inverse-cell_after', 'n': 50, 'exact': 44, 'listed_acceptable': 46, 'alternative_hit_ids': ['N076', 'N194']}
{'cohort': 'ordinary', 'model': 'none_after', 'n': 50, 'exact': 43, 'listed_acceptable': 45, 'alternative_hit_

## Takeaways

按具体错误原因改进数据，保留正常单字监督。先人工核实E/U，并补原文；为多种可接受切法建立独立语义验证。没有修改正式语料、重新训练或发布模型。完整逐条表与来源在 canonical report 和旁边 JSONL 中。

In [6]:
assert result == json.loads((REVIEW / 'findings.json').read_text())
print('Frozen inputs, independent counts, and report findings reconciled.')

Frozen inputs, independent counts, and report findings reconciled.
